In [10]:
INDEX = 0

# SETUP

In [11]:
import json

def read_jsonl(file_path):
    """
    Reads a JSON Lines (.jsonl) file and returns a list of Python dictionaries.
    
    Args:
        file_path (str): Path to the JSONL file.
    
    Returns:
        list: A list of dictionaries, one per line in the file.
    """
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                data.append(json.loads(line))
    return data
benchmark = read_jsonl("../../../benchmark/benchmark_archeology.jsonl")
benchmark[INDEX]

{'domain': 'archeology',
 'original_direct_question': 'What is the average Potassium in ppm from the first and last time the study recorded people in the Maltese area? Assume that Potassium is linearly interpolated between samples. Round your answer to 4 decimal places.',
 'answer': 8577.5298,
 'interactive_initial_prompt': '"I’m curious to dive into the historical data from the Maltese region. Could you help me get an overview of the different variables we have for past studies, like environmental factors, human activity, and any other available measurements? I’d like to understand how things like nutrient levels, especially minerals, evolved over time."',
 'static_initial_prompt': '"Could you pull up tables related to the Maltese region? I’m interested in any records on environmental measurements or human activity that might have been tracked over time. Specifically, I’d like to know what variables were recorded and their corresponding time points."'}

In [12]:
import bm25s
import Stemmer
stemmer = Stemmer.Stemmer("english")

In [13]:
DATA_SOURCES = ["archeology"]
INITIAL_PROMPT = benchmark[INDEX]["interactive_initial_prompt"]

In [14]:
def print_format_to_gpt(results):
    print("SYSTEM OUTPUT:")
    for result in results[0]:
        print(result)

In [15]:
def print_initial_prompt_to_chatgpt(domain: str, question: str):
    print(f"""You are simulating a domain expert in world cities, roman cities, radiocarbon data, world conflicts, and climate measurement exploration, who is interacting with a basic table discovery system to explore insights from an enterprise dataset.

This system supports static table lookup: it returns the contents (rows or description) of one or more tables based on your description. However:
- The system does not infer your deeper intent.
- The system does not combine, transform, or analyze data for you.
- The system does not explain its reasoning—it simply returns table contents for you to explore.

Your task is to gradually explore and refine your question about some aspect of the data. You do not begin with a precise question — your curiosity evolves step-by-step based on the tables you receive. You will gradually refine your question by examining the contents of the tables returned.

In this scenario:
- The system already has access to an internal dataset.
- You are familiar with the domain and have seen similar datasets before.
- You are not uploading new datasets or asking if data exists — you assume it does.

Here is a possible eventual goal (you do not know this at the start, and you may or may not reach it):

{question}

Your behavior should reflect:
- You are familiar with the domain but must infer relevant relationships from static tables.
- You refine your question step-by-step depending on what the returned tables show.
- You may explore tangents or ask for different table contents in later turns.
- You will only reach the specific question above if you deduce it from the table contents, which may take multiple turns.

Continue your role as the domain expert. This is the conversation so far (again, provide response as if you are prompting the system directly):

YOU: {INITIAL_PROMPT}""")

In [16]:
retriever = bm25s.BM25.load(f"indices/keyword-index-{DATA_SOURCES[0]}", load_corpus=True)

# TEST

In [17]:
print_initial_prompt_to_chatgpt(
    DATA_SOURCES[0], benchmark[INDEX]["original_direct_question"]
)

You are simulating a domain expert in world cities, roman cities, radiocarbon data, world conflicts, and climate measurement exploration, who is interacting with a basic table discovery system to explore insights from an enterprise dataset.

This system supports static table lookup: it returns the contents (rows or description) of one or more tables based on your description. However:
- The system does not infer your deeper intent.
- The system does not combine, transform, or analyze data for you.
- The system does not explain its reasoning—it simply returns table contents for you to explore.

Your task is to gradually explore and refine your question about some aspect of the data. You do not begin with a precise question — your curiosity evolves step-by-step based on the tables you receive. You will gradually refine your question by examining the contents of the tables returned.

In this scenario:
- The system already has access to an internal dataset.
- You are familiar with the doma

In [18]:
query_tokens = bm25s.tokenize(
    INITIAL_PROMPT, stemmer=stemmer, show_progress=False
)
results, _ = retriever.retrieve(query_tokens, k=10, show_progress=False)
print_format_to_gpt(results)

SYSTEM OUTPUT:
{'text': "Data Supplement to: 'A 3 million year index for North African humidity/aridity and the implication of potential pan-African Humid periods'; Note: the dust proxy is from Larrasoaña et al. (2003), rescaled to our chronology. It is based on magnetic measurements (IRM@0.9 AF 120mT (A/m)) reflecting hematite variations, which (at this site) reflect aeolian dust fluxes.", 'metadata': {'table': 'climateMeasurements_SEP_context_0'}}
{'text': 'Conflict: Two Sicilies (revolt with help from Aragon) | StartYear: 1282 | EndYear: 1284 | Fatalities: nan | Century: 1200 | Decade: 1280', 'metadata': {'table': 'conflict_brecke_SEP_content_836'}}
{'text': 'Conflict: Castile (with help from Moroccan Maranids) (rebellion) | StartYear: 1284 | EndYear: 1284 | Fatalities: nan | Century: 1200 | Decade: 1280', 'metadata': {'table': 'conflict_brecke_SEP_content_842'}}
{'text': 'city: Neder-Over-Heembeek | city_ascii: Neder-Over-Heembeek | lat: 50.9 | lng: 4.3833 | country: Belgium | iso2

In [19]:
query_tokens = bm25s.tokenize(
    """Thanks — that’s a helpful spread to start with. I see environmental proxies like magnetic measurements (possibly linked to dust fluxes), some historical conflict events, and a few cities with “Mineral” in the name (which are probably not directly relevant to Maltese mineral data, but I’ll keep them in mind).

To begin focusing more on the Maltese context — especially around human presence and potential mineral/nutrient levels — could you show me a table that includes radiocarbon dating or human activity markers in the Maltese region? I’d like to look at how early and late occupation or activity is recorded, and then I can start checking which environmental variables correspond to those time ranges.""",
    stemmer=stemmer,
    show_progress=False
)
results, _ = retriever.retrieve(query_tokens, k=10, show_progress=False)
print_format_to_gpt(results)

SYSTEM OUTPUT:
{'text': "Data Supplement to: 'A 3 million year index for North African humidity/aridity and the implication of potential pan-African Humid periods'; Note: the dust proxy is from Larrasoaña et al. (2003), rescaled to our chronology. It is based on magnetic measurements (IRM@0.9 AF 120mT (A/m)) reflecting hematite variations, which (at this site) reflect aeolian dust fluxes.", 'metadata': {'table': 'climateMeasurements_SEP_context_0'}}
{'text': 'city: Mineral Wells | city_ascii: Mineral Wells | lat: 32.8169 | lng: -98.0776 | country: United States | iso2: US | iso3: USA | admin_name: Texas | capital: nan | population: 14925.0 | id: 1840020689', 'metadata': {'table': 'worldcities_SEP_content_31313'}}
{'text': 'city: Mineral del Monte | city_ascii: Mineral del Monte | lat: 20.1333 | lng: -98.6667 | country: Mexico | iso2: MX | iso3: MEX | admin_name: Hidalgo | capital: minor | population: 14640.0 | id: 1484360018', 'metadata': {'table': 'worldcities_SEP_content_31329'}}
{'t

In [20]:
query_tokens = bm25s.tokenize(
    """Now, I’d like to pivot slightly to environmental or geochemical measurements that may correspond to this time span. Specifically, I’m looking for tables with mineral or elemental measurements (e.g., Potassium, Phosphorus, Iron) across time — preferably sediment core samples, soil layers, or any stratigraphic contexts.

Could you show me a table that contains mineral or nutrient measurements indexed by time or depth, ideally in the Maltese region or nearby Mediterranean areas?""",
    stemmer=stemmer,
    show_progress=False
)
results, _ = retriever.retrieve(query_tokens, k=10, show_progress=False)
print_format_to_gpt(results)

SYSTEM OUTPUT:
{'text': 'city: Time | city_ascii: Time | lat: 58.7228 | lng: 5.7653 | country: Norway | iso2: NO | iso3: NOR | admin_name: Rogaland | capital: nan | population: 19353.0 | id: 1578972107', 'metadata': {'table': 'worldcities_SEP_content_26592'}}
{'text': "Data Supplement to: 'A 3 million year index for North African humidity/aridity and the implication of potential pan-African Humid periods'; Note: the dust proxy is from Larrasoaña et al. (2003), rescaled to our chronology. It is based on magnetic measurements (IRM@0.9 AF 120mT (A/m)) reflecting hematite variations, which (at this site) reflect aeolian dust fluxes.", 'metadata': {'table': 'climateMeasurements_SEP_context_0'}}
{'text': 'city: Mineral Wells | city_ascii: Mineral Wells | lat: 32.8169 | lng: -98.0776 | country: United States | iso2: US | iso3: USA | admin_name: Texas | capital: nan | population: 14925.0 | id: 1840020689', 'metadata': {'table': 'worldcities_SEP_content_31313'}}
{'text': 'city: Mineral del Mont

In [21]:
query_tokens = bm25s.tokenize(
    """Could you return a table that includes geochemical variables — particularly Potassium (K), or other elements like Phosphorus (P), Calcium (Ca), Iron (Fe) — ideally indexed by either date (BP or BCE) or by sediment depth?

I’m especially interested in tables that may contain continuous measurements across layers, such as:
	•	Core samples
	•	Soil or peat stratigraphy
	•	Lake sediment profiles

If the dataset includes a location or site identifier, even better — I’ll try to link it to the Maltese region after that.""",
    stemmer=stemmer,
    show_progress=False
)
results, _ = retriever.retrieve(query_tokens, k=10, show_progress=False)
print_format_to_gpt(results)

SYSTEM OUTPUT:
{'text': 'Primary Key: Hanson2016_397 | Ancient Toponym: Lindinis | Modern Toponym: Ilchester | Province: Britannia | Country: United Kingdom | Barrington Atlas Rank: 4 or 5 | Barrington Atlas Reference: 8 E4 | Start Date: 100 | End Date: nan | Longitude (X): -2.687138 | Latitude (Y): 51.003593 | Select Bibliography: Millett 1990: Table 4.4; Millett 1990: Table 6.5; PECS; Wacher 1995.', 'metadata': {'table': 'roman_cities_SEP_content_396'}}
{'text': 'Primary Key: Hanson2016_401 | Ancient Toponym: Moridunum | Modern Toponym: Carmarthen | Province: Britannia | Country: United Kingdom | Barrington Atlas Rank: 3 | Barrington Atlas Reference: 8 C3 | Start Date: 100 | End Date: nan | Longitude (X): -4.296705 | Latitude (Y): 51.862004 | Select Bibliography: DGRG; Millett 1990: Table 4.4; Millett 1990: Table 6.5; PECS; Wacher 1995.', 'metadata': {'table': 'roman_cities_SEP_content_400'}}
{'text': 'Primary Key: Hanson2016_406 | Ancient Toponym: Venta Icenorum | Modern Toponym: Ca

In [22]:
query_tokens = bm25s.tokenize(
    """Could you show me a table that contains elemental concentration values across samples, such as Potassium (K), Phosphorus (P), or others, ideally alongside:
	•	Sample depth or sample year (BP or BCE),
	•	And possibly a site or core identifier?

These might come from:
	•	Sediment cores
	•	Paleosols
	•	Geochemical profiles
	•	Any long-term paleoenvironmental reconstruction work in the Mediterranean or North Africa

Even if it’s not Maltese specifically, I’d like to see the structure of the geochemical tables available so I can begin anchoring Potassium trends to radiocarbon periods.""",
    stemmer=stemmer,
    show_progress=False
)
results, _ = retriever.retrieve(query_tokens, k=10, show_progress=False)
print_format_to_gpt(results)

SYSTEM OUTPUT:
{'text': "Data Supplement to: 'A 3 million year index for North African humidity/aridity and the implication of potential pan-African Humid periods'; Note: the dust proxy is from Larrasoaña et al. (2003), rescaled to our chronology. It is based on magnetic measurements (IRM@0.9 AF 120mT (A/m)) reflecting hematite variations, which (at this site) reflect aeolian dust fluxes.", 'metadata': {'table': 'climateMeasurements_SEP_context_0'}}
{'text': 'Region: Central Italy | Site: Isola Santa, Careggina | Layer/Assemblage/Context: nan | Culture: Mesolithic | Lab code: R-1529a | ProjCode: 401 | date: 9,220 | error: 100 | Material: Charcoal | Species: nan | Marine?: nan | ΔR: nan | ΔR error: nan | % marine: nan | Cal. BC 1 sigma: nan | Cal. BC 1 sigma.1: nan | Cal. BC 2 sigma: nan | Cal. BC 2 sigma.1: nan | Latitude: 10.3124 | Longitude: 44.064900 | Notes: nan | reference: EUROEVOL-Sample:1781', 'metadata': {'table': 'radiocarbon_database_regional_SEP_content_400'}}
{'text': 'Reg

In [23]:
query_tokens = bm25s.tokenize(
    """Please return a table that includes Potassium (K) concentrations, listed in ppm (parts per million), alongside either:
	•	Sample year (BP or BCE) or
	•	Sample depth (in cm or m)

It can also include other variables (Phosphorus, Iron, Calcium, etc.) — but Potassium must be one of them. If available, I’d also like to see a site identifier or location coordinates to help me situate the data.

Even if it’s not from Malta, I need to inspect how Potassium is structured and measured over time.""",
    stemmer=stemmer,
    show_progress=False
)
results, _ = retriever.retrieve(query_tokens, k=10, show_progress=False)
print_format_to_gpt(results)

SYSTEM OUTPUT:
{'text': "Data Supplement to: 'A 3 million year index for North African humidity/aridity and the implication of potential pan-African Humid periods'; Note: the dust proxy is from Larrasoaña et al. (2003), rescaled to our chronology. It is based on magnetic measurements (IRM@0.9 AF 120mT (A/m)) reflecting hematite variations, which (at this site) reflect aeolian dust fluxes.", 'metadata': {'table': 'climateMeasurements_SEP_context_0'}}
{'text': 'Conflict: Two Sicilies (revolt with help from Aragon) | StartYear: 1282 | EndYear: 1284 | Fatalities: nan | Century: 1200 | Decade: 1280', 'metadata': {'table': 'conflict_brecke_SEP_content_836'}}
{'text': 'Conflict: Castile (with help from Moroccan Maranids) (rebellion) | StartYear: 1284 | EndYear: 1284 | Fatalities: nan | Century: 1200 | Decade: 1280', 'metadata': {'table': 'conflict_brecke_SEP_content_842'}}
{'text': 'city: Time | city_ascii: Time | lat: 58.7228 | lng: 5.7653 | country: Norway | iso2: NO | iso3: NOR | admin_nam

In [24]:
query_tokens = bm25s.tokenize(
    """Could you return any table with continuous numeric measurements by depth or year — even if they aren’t explicitly labeled as Potassium? I’m looking for structured environmental proxies that might resemble geochemical data. Specifically:
	•	Column names like depth_cm, ppm, mg/kg, element, or measurement_type
	•	Or even unlabeled numeric series that might later be interpreted as geochemical

This will help confirm whether any numeric stratigraphic measurement data exists — even if not properly labeled. If nothing turns up, I’ll shift strategy.""",
    stemmer=stemmer,
    show_progress=False
)
results, _ = retriever.retrieve(query_tokens, k=10, show_progress=False)
print_format_to_gpt(results)

SYSTEM OUTPUT:
{'text': "Data Supplement to: 'A 3 million year index for North African humidity/aridity and the implication of potential pan-African Humid periods'; Note: the dust proxy is from Larrasoaña et al. (2003), rescaled to our chronology. It is based on magnetic measurements (IRM@0.9 AF 120mT (A/m)) reflecting hematite variations, which (at this site) reflect aeolian dust fluxes.", 'metadata': {'table': 'climateMeasurements_SEP_context_0'}}
{'text': "city: Piešťany | city_ascii: Piest'any | lat: 48.5842 | lng: 17.8336 | country: Slovakia | iso2: SK | iso3: SVK | admin_name: Trnava | capital: minor | population: 27681.0 | id: 1703636029", 'metadata': {'table': 'worldcities_SEP_content_21086'}}
{'text': 'city: Proper Bansud | city_ascii: Proper Bansud | lat: 12.8594 | lng: 121.4567 | country: Philippines | iso2: PH | iso3: PHL | admin_name: Oriental Mindoro | capital: nan | population: 42671.0 | id: 1608041757', 'metadata': {'table': 'worldcities_SEP_content_16740'}}
{'text': 'c